# 🌿 Aura Farming — Phase 4: MobileNetV3 Training on Colab

**An AI-Powered, Reliability-Aware Crop Disease Detection System**

> *Smarter Farming. Healthier Crops.*

---

## What this notebook does

This notebook trains the MobileNetV3 Small baseline model for Aura Farming's
tomato disease detection system.

**Two-phase training:**
- **Phase A** — Feature extraction: freeze the MobileNetV3 base, train the new 10-class head only.
- **Phase B** — Fine-tuning (optional): unfreeze the upper base layers and continue at a lower learning rate.

**Safeguards built in:**
- The test manifest (`test_manifest.csv`) is **never loaded**. It is reserved for Phase 7 final evaluation.
- No predictions are fabricated. If training fails, the notebook stops and explains why.
- Class weights are computed from the **training split only**.

---

## Before you start — checklist

1. ✅ Runtime type is set to **GPU** → *Runtime > Change runtime type > T4 GPU*
2. ✅ You have uploaded the project to Google Drive (see Section 1)
3. ✅ You have uploaded the PlantVillage dataset to Google Drive (see Section 1)
4. ✅ You have **not** modified the split manifests

**Expected total training time on a free T4 GPU:**
- Phase A only: ~15–25 minutes (up to 30 epochs with early stopping)
- Phase A + Phase B: ~30–45 minutes


---
## Section 1: Mount Google Drive and set paths

You need to:
1. Upload the entire `Aura Farming/` project folder to your Google Drive.
2. Upload the PlantVillage color dataset — specifically the `color/` folder
   (containing all `Tomato___*` subfolders) to your Google Drive.

**Recommended Drive structure:**
```
MyDrive/
  AuraFarming/                ← your project folder
    configs/
    data/
      plantvillage/
        color/                ← the color/ folder from the repo clone
          Tomato___Bacterial_spot/
          Tomato___Early_blight/
          ... (all 10 class folders)
      splits/
        train_manifest.csv
        val_manifest.csv
        test_manifest.csv
        split_report.json
    models/
    reports/
    src/
    training/
```

> **Tip:** The fastest upload method is to zip `Aura Farming/` on your Mac,
> upload the zip to Drive, then unzip it in a Colab cell:
> `!unzip /content/drive/MyDrive/AuraFarming.zip -d /content/drive/MyDrive/`


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# ── Configure these two paths to match your Drive layout ──────────

# Where you placed the 'Aura Farming' project folder on Google Drive:
PROJECT_ROOT_COLAB = '/content/drive/MyDrive/AuraFarming'

# Where the PlantVillage color/ folder lives on Colab
# (the folder that directly contains the Tomato___ subfolders):
DATASET_COLOR_DIR = f'{PROJECT_ROOT_COLAB}/data/plantvillage/color'

# ── Verify the paths exist ────────────────────────────────────────
import os

for label, path in [
    ('Project root',      PROJECT_ROOT_COLAB),
    ('Dataset color dir', DATASET_COLOR_DIR),
    ('Train manifest',    f'{PROJECT_ROOT_COLAB}/data/splits/train_manifest.csv'),
    ('Val manifest',      f'{PROJECT_ROOT_COLAB}/data/splits/val_manifest.csv'),
]:
    exists = os.path.exists(path)
    status = '✅' if exists else '❌  NOT FOUND — check your Drive path'
    print(f'{status}  {label}: {path}')

# Confirm test manifest is NOT loaded (just check it exists for reference)
test_manifest_path = f'{PROJECT_ROOT_COLAB}/data/splits/test_manifest.csv'
print(f'\nTest manifest present: {os.path.exists(test_manifest_path)}')
print('  (Not loaded in this notebook — reserved for Phase 7 evaluation)')

---
## Section 2: Install dependencies

Colab already includes TensorFlow and NumPy. We only install the small
packages that are not pre-installed.

> **Do not** install `tensorflow` or `numpy` here — Colab's pre-installed
> versions are already correctly configured for the T4 GPU.


In [ ]:
# Install only the packages not pre-installed in Colab
!pip install --quiet \
    "PyYAML>=6.0.1,<7.0.0" \
    "Pillow>=10.2.0,<11.0.0" \
    "scikit-learn>=1.4.0,<2.0.0" \
    "seaborn>=0.13.0,<1.0.0"

print('✅  Extra packages installed.')

---
## Section 3: Environment check

Verifies Python version, TensorFlow version, NumPy version, and GPU availability.

Expected output:
- Python 3.12.x
- TensorFlow 2.18–2.20 (Colab's current version)
- NumPy 2.x
- GPU: ✅ NVIDIA T4 (or similar)


In [ ]:
import sys
sys.path.insert(0, PROJECT_ROOT_COLAB)

from training.colab_utils import check_colab_environment

env_info = check_colab_environment(verbose=True)

if not env_info['gpu_available']:
    print('\n⚠  WARNING: No GPU detected.')
    print('   Training on CPU will take hours instead of minutes.')
    print('   Go to Runtime > Change runtime type > Hardware accelerator > T4 GPU')
    print('   then re-run from Section 1.')
else:
    print('\n✅  GPU detected. Training will be fast.')

---
## Section 4: Remap manifest paths

The split manifests were generated on a Mac and contain absolute paths like:
```
/Users/giftarachel/Aura Farming/data/plantvillage/color/Tomato___X/img.JPG
```

This cell remaps them to the Colab dataset directory.
The original CSV files are **not modified** — remapping happens in memory only.
Class labels, split assignments, and group IDs are preserved exactly.


In [ ]:
from pathlib import Path
from training.colab_utils import remap_manifest_paths, verify_remapped_paths

splits_dir = Path(PROJECT_ROOT_COLAB) / 'data' / 'splits'
dataset_root = Path(DATASET_COLOR_DIR)

# Remap training paths
print('Remapping train manifest...')
train_paths, train_labels = remap_manifest_paths(
    splits_dir / 'train_manifest.csv',
    dataset_root,
)

# Remap validation paths
print('Remapping val manifest...')
val_paths, val_labels = remap_manifest_paths(
    splits_dir / 'val_manifest.csv',
    dataset_root,
)

# NOTE: test_manifest.csv is intentionally NOT remapped or loaded here.
# It is reserved for Phase 7 final evaluation.

print(f'\nTraining images  : {len(train_paths)}')
print(f'Validation images: {len(val_paths)}')
print(f'Test set         : NOT loaded (reserved for Phase 7)')

# Verify a sample of paths actually exist on disk
print('\nVerifying training paths...')
train_ok, train_missing = verify_remapped_paths(train_paths, sample_size=20)

print('Verifying validation paths...')
val_ok, val_missing = verify_remapped_paths(val_paths, sample_size=10)

if not (train_ok and val_ok):
    raise RuntimeError(
        'Some image paths could not be found.\n'
        'Check that DATASET_COLOR_DIR in Section 1 points to the folder '
        'that directly contains the Tomato___ class subfolders.'
    )

print('\n✅  All sampled paths found. Ready to train.')

---
## Section 5: Smoke test

Before committing to full training, this cell:
1. Loads **one small batch** (8 images) from the training set.
2. Builds MobileNetV3 Small with a 10-class head.
3. Runs **one forward + backward pass** to confirm everything is wired correctly.

If this cell completes without error, the full training pipeline is ready.
If it fails, read the error message and fix the issue before proceeding.


In [ ]:
import os, random, time
import numpy as np
import tensorflow as tf
from tensorflow.keras.applications import MobileNetV3Small

# NOTE: We do NOT import preprocess_input from mobilenet_v3.
# In Keras 3 (TF >= 2.16) it became a no-op. We rescale manually.
from tensorflow.keras import layers, Model
from tensorflow.keras.optimizers import Adam

# ── Reproducibility ───────────────────────────────────────────────
SEED = 42
os.environ['PYTHONHASHSEED'] = str(SEED)
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

# ── Constants ─────────────────────────────────────────────────────
NUM_CLASSES  = 10
INPUT_SIZE   = (224, 224)
SMOKE_BATCH  = 8
H, W         = INPUT_SIZE

print('=' * 55)
print('  Aura Farming — Smoke Test')
print(f'  TensorFlow : {tf.__version__}')
print(f'  GPU        : {tf.config.list_physical_devices("GPU")}')
print('=' * 55)

# ── Step 1: Load a small batch of real training images ────────────
print('\nStep 1: Loading a small batch of training images...')

# Pick 8 training images (2 from different classes for variety)
from collections import defaultdict
by_class = defaultdict(list)
for path, label in zip(train_paths[:500], train_labels[:500]):
    by_class[label].append(path)

smoke_paths, smoke_labels = [], []
for cls in sorted(by_class.keys())[:SMOKE_BATCH]:
    smoke_paths.append(by_class[cls][0])
    smoke_labels.append(cls)

def _load_image(path: str) -> np.ndarray:
    """Load one image, resize to 224×224, apply MobileNetV3 preprocessing."""
    raw = tf.io.read_file(path)
    img = tf.image.decode_image(raw, channels=3, expand_animations=False)
    img = tf.image.resize(img, [H, W])
    img = tf.cast(img, tf.float32)
    img = img / 127.5 - 1.0  # Manual MobileNetV3 rescaling: [0,255] → [-1,1]
    return img.numpy()

batch_images = np.stack([_load_image(p) for p in smoke_paths])
batch_labels = np.array(smoke_labels, dtype=np.int32)

print(f'  Batch shape  : {batch_images.shape}   (expected: ({SMOKE_BATCH}, 224, 224, 3))')
print(f'  Pixel range  : [{batch_images.min():.3f}, {batch_images.max():.3f}]  (expected: [-1, 1])')
print(f'  Labels       : {batch_labels.tolist()}')

assert batch_images.shape == (SMOKE_BATCH, H, W, 3), 'Wrong batch shape'
assert batch_images.min() >= -1.0 - 1e-4, 'Pixels below -1'
assert batch_images.max() <= 1.0 + 1e-4, 'Pixels above +1'
print('  ✅ Batch loaded and preprocessed correctly.')

# ── Step 2: Build the model ───────────────────────────────────────
print('\nStep 2: Building MobileNetV3 Small...')

inputs = tf.keras.Input(shape=(H, W, 3), name='input_image')
base = MobileNetV3Small(
    input_shape=(H, W, 3),
    include_top=False,
    weights='imagenet',
    include_preprocessing=False,  # we rescale manually in the data pipeline
    name='mobilenetv3small',       # explicit name for get_layer() lookups
)
base.trainable = False

x = base(inputs, training=False)
x = layers.GlobalAveragePooling2D(name='gap')(x)
x = layers.BatchNormalization(name='bn_head')(x)
x = layers.Dropout(0.3, name='dropout_head')(x)
outputs = layers.Dense(NUM_CLASSES, activation='softmax', name='predictions')(x)

model = Model(inputs, outputs, name='aura_farming_mobilenetv3')
model.compile(
    optimizer=Adam(learning_rate=1e-3),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy'],
)

trainable = sum(p.numpy().size for p in model.trainable_variables)
total     = sum(p.numpy().size for p in model.variables)
print(f'  Trainable params : {trainable:,}')
print(f'  Total params     : {total:,}')
print(f'  Base frozen      : {not base.trainable}')
print('  ✅ Model built successfully.')

# ── Step 3: One forward pass ──────────────────────────────────────
print('\nStep 3: Forward pass...')
t0 = time.time()
probs = model(batch_images, training=False).numpy()
elapsed = time.time() - t0

assert probs.shape == (SMOKE_BATCH, NUM_CLASSES), f'Wrong output shape: {probs.shape}'
np.testing.assert_allclose(probs.sum(axis=1), 1.0, atol=1e-5)
print(f'  Output shape   : {probs.shape}   (expected: ({SMOKE_BATCH}, {NUM_CLASSES}))')
print(f'  Probs sum≈1    : ✅')
print(f'  Inference time : {elapsed*1000:.0f} ms for {SMOKE_BATCH} images')

# ── Step 4: One training step ─────────────────────────────────────
print('\nStep 4: One training step (gradient update)...')
t0 = time.time()
history_smoke = model.fit(
    batch_images,
    batch_labels,
    batch_size=SMOKE_BATCH,
    epochs=1,
    verbose=0,
)
elapsed = time.time() - t0
smoke_loss = history_smoke.history['loss'][0]
smoke_acc  = history_smoke.history['accuracy'][0]
print(f'  Smoke loss     : {smoke_loss:.4f}')
print(f'  Smoke accuracy : {smoke_acc:.4f}')
print(f'  Step time      : {elapsed:.2f}s')

assert not np.isnan(smoke_loss), 'Loss is NaN — check preprocessing'

# ── Summary ───────────────────────────────────────────────────────
print('\n' + '=' * 55)
print('  ✅  Smoke test PASSED')
print('  All checks passed. Ready for full training.')
print('  Run Section 6 to start Phase A training.')
print('=' * 55)

# Clean up smoke test model to free memory before full training
del model, base, batch_images
import gc; gc.collect()

---
## Section 6: Full Phase A training — Feature Extraction

**What happens here:**
- Load MobileNetV3 Small with ImageNet weights.
- Freeze the entire convolutional base.
- Train only the new 10-class classification head.
- Use class weights (computed from training set only) to handle imbalance.
- Apply early stopping when `val_loss` stops improving (patience = 5 epochs).
- Save the best model to `models/mobilenetv3_phase_a.keras`.
- Evaluate on the **validation set only** (test set is never touched).

**Expected duration:** ~15–25 minutes on a T4 GPU.

After Phase A, proceed to Section 7 to run Phase B fine-tuning (optional but recommended).


In [ ]:
import sys
sys.path.insert(0, PROJECT_ROOT_COLAB)

from training.train_baseline import (
    load_config,
    load_class_mapping,
    set_seeds,
    compute_class_weights,
    build_tf_dataset,
    build_model,
    build_callbacks,
    save_history,
    evaluate_on_validation,
    update_config_weights_path,
)

cfg     = load_config()
mapping = load_class_mapping()

SEED        = cfg['dataset']['random_seed']
INPUT_SIZE  = tuple(cfg['model']['input_size'])
NUM_CLASSES = len(mapping['classes'])
class_names = {int(k): v['display_name'] for k, v in mapping['classes'].items()}

trn_cfg     = cfg['training']
BATCH_SIZE  = trn_cfg['batch_size']
EPOCHS_A    = trn_cfg['phase_a_epochs']
PATIENCE_A  = trn_cfg['phase_a_patience']
DROPOUT     = trn_cfg['dropout_rate']
AUGMENT     = trn_cfg['augment']

# Use PROJECT_ROOT_COLAB for all output paths
from pathlib import Path
project_root = Path(PROJECT_ROOT_COLAB)
models_dir   = project_root / 'models'
reports_dir  = project_root / 'reports'
models_dir.mkdir(parents=True, exist_ok=True)
reports_dir.mkdir(parents=True, exist_ok=True)

# Seeds
set_seeds(SEED)
import tensorflow as tf
tf.random.set_seed(SEED)

print('=' * 60)
print('  Aura Farming — Phase A: Feature Extraction')
print(f'  Input size  : {INPUT_SIZE}')
print(f'  Classes     : {NUM_CLASSES}')
print(f'  Batch size  : {BATCH_SIZE}')
print(f'  Max epochs  : {EPOCHS_A}  (patience={PATIENCE_A})')
print(f'  Augment     : {AUGMENT}')
print(f'  Seed        : {SEED}')
print('=' * 60)

# Class weights from TRAINING set only
class_weights = compute_class_weights(train_labels, NUM_CLASSES)

# Data pipelines  (train_paths / val_paths from Section 4)
print('Building data pipelines...')
train_ds = build_tf_dataset(
    train_paths, train_labels, INPUT_SIZE,
    batch_size=BATCH_SIZE, augment=AUGMENT,
    shuffle=True, seed=SEED,
)
val_ds = build_tf_dataset(
    val_paths, val_labels, INPUT_SIZE,
    batch_size=BATCH_SIZE, augment=False,
    shuffle=False, seed=SEED,
)

# Model
model_a = build_model(NUM_CLASSES, INPUT_SIZE, DROPOUT)
model_a.summary()

# Callbacks — checkpoint saves to Google Drive automatically
ckpt_a = models_dir / 'mobilenetv3_phase_a.keras'
callbacks_a = build_callbacks(ckpt_a, PATIENCE_A, 'phase_a')

# Training
print('\nStarting Phase A training...')
history_a = model_a.fit(
    train_ds,
    validation_data=val_ds,
    epochs=EPOCHS_A,
    callbacks=callbacks_a,
    class_weight=class_weights,
    verbose=1,
)

print(f'\n✅  Phase A complete. Epochs run: {len(history_a.epoch)}')
print(f'   Best model saved to: {ckpt_a}')

# Save history
save_history(history_a, reports_dir, 'phase_a')

# Evaluate on validation set only
metrics_a = evaluate_on_validation(
    model_a, val_ds, val_labels, class_names,
    output_dir=reports_dir, phase_label='phase_a',
)

print(f'\n  Validation accuracy : {metrics_a["val_accuracy"]:.4f}')
print(f'  Macro F1            : {metrics_a["macro_f1"]:.4f}')
print('  (These are genuine results from the validation set.)')
print('  (Test set has NOT been used — reserved for Phase 7.)')

---
## Section 7: Phase B — Fine-Tuning (recommended)

**What happens here:**
- Unfreeze the upper layers of the MobileNetV3 base (from layer 100 onward).
- Continue training at a much lower learning rate (1e-5) to avoid destroying
  the ImageNet features that were already learned.
- Save the best fine-tuned model to `models/mobilenetv3_baseline.keras`.

**Why fine-tune?**
The ImageNet features in the base model are very useful, but they were learned
from general images (not plant leaves). Carefully fine-tuning the upper layers
allows the model to adapt to tomato leaf textures and disease patterns while
keeping the foundational feature detectors intact.

**Expected duration:** Additional ~15–20 minutes on T4.

**Prerequisite:** Run Section 6 first.


In [ ]:
from training.train_baseline import unfreeze_for_fine_tuning

FINE_TUNE_AT = trn_cfg['fine_tune_at']    # 100 — unfreeze from layer 100 onward
EPOCHS_B     = trn_cfg['phase_b_epochs']  # 20
PATIENCE_B   = trn_cfg['phase_b_patience'] # 7
LR_B         = trn_cfg['phase_b_lr']       # 1e-5

print('=' * 60)
print('  Aura Farming — Phase B: Fine-Tuning')
print(f'  Unfreeze from layer: {FINE_TUNE_AT}')
print(f'  Learning rate      : {LR_B}')
print(f'  Max epochs         : {EPOCHS_B}  (patience={PATIENCE_B})')
print('=' * 60)

unfreeze_for_fine_tuning(model_a, FINE_TUNE_AT, learning_rate=LR_B)

ckpt_b = models_dir / 'mobilenetv3_baseline.keras'
callbacks_b = build_callbacks(ckpt_b, PATIENCE_B, 'phase_b')

print('\nStarting Phase B fine-tuning...')
history_b = model_a.fit(
    train_ds,
    validation_data=val_ds,
    epochs=EPOCHS_B,
    callbacks=callbacks_b,
    class_weight=class_weights,
    verbose=1,
)

print(f'\n✅  Phase B complete. Epochs run: {len(history_b.epoch)}')
print(f'   Best model saved to: {ckpt_b}')

save_history(history_b, reports_dir, 'phase_b')

metrics_b = evaluate_on_validation(
    model_a, val_ds, val_labels, class_names,
    output_dir=reports_dir, phase_label='phase_b',
)

print(f'\n  Validation accuracy : {metrics_b["val_accuracy"]:.4f}')
print(f'  Macro F1            : {metrics_b["macro_f1"]:.4f}')
print('  (Test set has NOT been used — reserved for Phase 7.)')

---
## Section 8: Update config.yaml and download model files

After training completes, this cell:
1. Updates `configs/config.yaml` so the Streamlit app knows where the model is.
2. Lists the output files you should download back to your Mac.


In [ ]:
import json

# Determine which model to use (Phase B if it exists, otherwise Phase A)
if ckpt_b.exists():
    final_model = ckpt_b
    print(f'Using Phase B (fine-tuned) model: {ckpt_b}')
else:
    final_model = ckpt_a
    print(f'Using Phase A model (Phase B not run): {ckpt_a}')

# Update config.yaml in the Drive copy
import re
config_path = project_root / 'configs' / 'config.yaml'
with open(config_path) as f:
    config_text = f.read()

# The model path should be relative to project root
rel_model_path = final_model.relative_to(project_root)
config_text = re.sub(
    r'weights_path:.*',
    f'weights_path: "{rel_model_path}"',
    config_text,
)
with open(config_path, 'w') as f:
    f.write(config_text)

print(f'✅  config.yaml updated: weights_path = {rel_model_path}')
print()

# List files to download back to your Mac
print('=' * 60)
print('  Files to download back to your Mac')
print('=' * 60)
files_to_download = [
    final_model,
    models_dir / 'mobilenetv3_phase_a.keras',
    reports_dir / 'val_metrics_phase_a.json',
    reports_dir / 'val_metrics_phase_b.json',
    reports_dir / 'training_history_phase_a.json',
    reports_dir / 'training_history_phase_b.json',
    reports_dir / 'confusion_matrix_phase_a.png',
    reports_dir / 'confusion_matrix_phase_b.png',
    reports_dir / 'training_log_phase_a.csv',
    reports_dir / 'training_log_phase_b.csv',
    config_path,
]

import os
for f in files_to_download:
    exists = os.path.exists(f)
    status = '✅' if exists else '⬜ (not generated)'
    print(f'  {status}  {f}')

print()
print('Place downloaded files in the same relative paths under your local')
print('"Aura Farming/" folder. Then run the Streamlit app normally.')
print()
print('⚠  Do NOT evaluate on test_manifest.csv until Phase 7.')

---
## Section 9: Troubleshooting

### ❌ `FileNotFoundError` on image paths
The most common issue. Double-check that `DATASET_COLOR_DIR` in Section 1
points to the folder that **directly** contains the `Tomato___*` subfolders.

Run this to list what's in your dataset directory:
```python
import os
print(os.listdir(DATASET_COLOR_DIR)[:15])
```
You should see names like `Tomato___Bacterial_spot`, `Tomato___healthy`, etc.

### ❌ `RuntimeError: Drive not mounted`
Re-run Section 1 and click the Google Drive authentication link when prompted.

### ❌ `CUDA out of memory`
Reduce batch size: change `batch_size: 32` to `batch_size: 16` in
`configs/config.yaml`, then restart from Section 6.

### ⚠ Training is very slow (>5 min per epoch)
You are running on CPU. Go to **Runtime → Change runtime type →
Hardware accelerator → T4 GPU**, then restart from Section 1.

### ❌ `ImportError: No module named 'yaml'` or `'sklearn'`
Re-run Section 2 (the pip install cell).

### ❌ `AttributeError: ...include_preprocessing`
Colab's TF version changed the API. Run this to check:
```python
import tensorflow as tf; print(tf.__version__)
```
If TF < 2.15, upgrade: `!pip install -q tensorflow --upgrade`
then restart the runtime and run from Section 1.
